# atari_endpoint_replicate

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each game's stored DDQN runs and coresets' published DDQN curves.
COMPONENT = "ddqn_atari"
component = EXPERIMENT.component(COMPONENT)
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP

results = {
    game: load_results(EXPERIMENT, COMPONENT, where={"ENV_HYPERS.GAME": game})
    for game in GAMES
}
results = {game: runs for game, runs in results.items() if runs is not None}
reference = json.loads(Path("coresets_ddqn_1m_reference.json").read_text())

In [ ]:
# Compute each game's mean return over frames, and its reference up to our last frame.
curves = {}
for game, runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    ref_frames = np.asarray(reference[game]["frames"])
    ref_return = np.asarray(reference[game]["mean_return"])
    in_window = ref_frames <= TOTAL_FRAMES
    ref_curve = (ref_frames[in_window], ref_return[in_window])
    curves[game] = (timesteps * FRAMESKIP, mean_ci(returns), ref_curve)

In [ ]:
# Style for the per-game learning-curve plots.
GAME_STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "legend.loc": "upper left",
}
DDQN_LINE = {"color": "tab:blue", "linewidth": 1.75, "label": "DDQN"}
BAND_ALPHA = 0.2
REFERENCE_LINE = {"color": "black", "linewidth": 2.5, "label": "Endpoint Ref"}
XLABEL = "Frames"
YLABEL = "Return"
YLABEL_KW = {"va": "center", "labelpad": 20}
XTICKS = np.array([10, 20, 30, 40, 50]) * 1_000_000
XTICK_LABELS = [f"{tick // 1_000_000}M" for tick in XTICKS]


def title(game):
    return game.replace("_", " ").title()

In [ ]:
# Plot one learning curve per game against the coresets reference.
def draw(ax, game):
    frames, (mean, low, high), ref_curve = curves[game]
    ax.fill_between(frames, low, high, color=DDQN_LINE["color"], alpha=BAND_ALPHA)
    ax.plot(frames, mean, **DDQN_LINE)
    ax.plot(*ref_curve, **REFERENCE_LINE)


figures = {}
for game in curves:
    with plt.rc_context(GAME_STYLE):
        fig, ax = plt.subplots()
        draw(ax, game)
        ax.set(title=title(game), xlabel=XLABEL)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.legend()
        fig.tight_layout()
    figures[f"{COMPONENT}_{game}"] = fig
plt.show()

In [ ]:
# Style for the all-games grid.
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.035), "ncol": 2}

In [ ]:
# Plot every game in one grid against the coresets reference.
grids = {}
if curves:
    with plt.rc_context(GRID_STYLE):
        fig, axes = plt.subplots(*GRID_SHAPE)
        legend = {}
        for game, (column, row) in PANELS.items():
            ax = axes[row, column]
            ax.set_box_aspect(PANEL_ASPECT)
            if game not in curves:
                ax.axis("off")
                continue
            draw(ax, game)
            ax.set_title(title(game))
            ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
            handles, labels = ax.get_legend_handles_labels()
            legend.update(zip(labels, handles, strict=True))
        fig.tight_layout(**GRID_LAYOUT)
        fig.supxlabel(**GRID_XLABEL)
        fig.supylabel(**GRID_YLABEL)
        fig.legend(legend.values(), legend.keys(), **GRID_LEGEND)
    grids["grid"] = fig
figures.update(grids)
plt.show()

In [ ]:
# Save every plotted figure as a PDF, and the grid also as a PNG, in the plots dir.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
for name, fig in grids.items():
    fig.savefig(PLOTS_DIR / f"{name}.png", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")